# AquaCart — 3D Logo Intro · Colab Render

Renders `aquacart_logo_intro.blend` (120 frames, 30 fps, 1080×1080) on a Colab GPU.

**Before you start:** `Runtime ▸ Change runtime type ▸ T4 GPU`. Without a GPU this
falls back to CPU and will take hours.

Upload two files to this session: `aquacart_logo_intro.blend` and `render_colab.py`.

In [ ]:
#@title 1 · Check the GPU
!nvidia-smi || echo "NO GPU — set Runtime > Change runtime type > T4 GPU, then rerun."

In [ ]:
#@title 2 · Install Blender 5.2 LTS (~2 min)
BLENDER_URL = "https://download.blender.org/release/Blender5.2/blender-5.2.0-linux-x64.tar.xz"

import os, urllib.request
if not os.path.exists("/content/blender/blender"):
    !wget -q --show-progress -O /content/blender.tar.xz {BLENDER_URL}
    !mkdir -p /content/blender && tar -xf /content/blender.tar.xz -C /content/blender --strip-components=1
    !rm -f /content/blender.tar.xz
!/content/blender/blender --version | head -2

In [ ]:
#@title 3 · Upload the .blend and the render script
from google.colab import files
import os

os.makedirs("/content/frames", exist_ok=True)
missing = [f for f in ("aquacart_logo_intro.blend", "render_colab.py")
           if not os.path.exists("/content/" + f)]
if missing:
    print("Select:", ", ".join(missing))
    up = files.upload()
    for name in up:
        os.replace(name, "/content/" + name)
print(sorted(os.listdir("/content")))

### Engine note

The scene is lit and tuned for **EEVEE**, which needs a GL context that headless
Colab usually cannot provide. `--engine auto` tries EEVEE, and falls back to
**Cycles on GPU** with the lamp energies rebalanced (Cycles reads the same
wattages much hotter).

Cycles output will look slightly different from the local EEVEE preview —
softer shadows, more accurate glossy reflections. That is expected, and generally
better for a hero logo render.

Force one or the other with `--engine cycles` / `--engine eevee`.

In [ ]:
#@title 4 · Render the sequence
ENGINE  = "auto"   #@param ["auto", "cycles", "eevee"]
SAMPLES = 128      #@param {type:"integer"}
RES     = 1080     #@param {type:"integer"}
START   = 1        #@param {type:"integer"}
END     = 120      #@param {type:"integer"}

!/content/blender/blender -b /content/aquacart_logo_intro.blend     -P /content/render_colab.py --     --engine {ENGINE} --samples {SAMPLES} --res {RES}     --start {START} --end {END} --out /content/frames/

In [ ]:
#@title 5 · Encode to MP4 + GIF
!ls /content/frames | wc -l
!ffmpeg -y -loglevel error -framerate 30 -i /content/frames/f_%04d.png     -c:v libx264 -pix_fmt yuv420p -crf 16 -movflags +faststart     /content/aquacart_logo_intro.mp4

# transparent-background WebM, handy for overlaying on a website hero
!ffmpeg -y -loglevel error -framerate 30 -i /content/frames/f_%04d.png     -c:v libvpx-vp9 -pix_fmt yuva420p -crf 24 -b:v 0     /content/aquacart_logo_intro.webm

!ffmpeg -y -loglevel error -i /content/aquacart_logo_intro.mp4     -vf "fps=24,scale=480:-1:flags=lanczos,split[a][b];[a]palettegen[p];[b][p]paletteuse"     /content/aquacart_logo_intro.gif
!ls -lh /content/*.mp4 /content/*.webm /content/*.gif

In [ ]:
#@title 6 · Preview inline
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open("/content/aquacart_logo_intro.mp4","rb").read()).decode()
HTML(f'<video width=480 controls autoplay loop><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

In [ ]:
#@title 7 · Download (or save to Drive)
from google.colab import files
files.download("/content/aquacart_logo_intro.mp4")

# Keep the frames too — Colab wipes the session when it disconnects:
# from google.colab import drive; drive.mount("/content/drive")
# !cp /content/aquacart_logo_intro.* /content/drive/MyDrive/
# !zip -qr /content/drive/MyDrive/aquacart_frames.zip /content/frames